In [ ]:
import numpy as np
from scipy.stats import norm # pour la cdf de la loi normale
from math import factorial, exp, sqrt, log

# Formules de Black-Scholes

In [5]:
def black_scholes_call(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return max(S - K * exp(-r * T), 0.0)
    d1 = (log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return S * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)

def black_scholes_put(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return max(K * exp(-r * T) - S, 0.0)
    d1 = (log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return K * exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def black_scholes(S, K, T, r, sigma, option_type='call'):
    if option_type == 'call':
        return black_scholes_call(S, K, T, r, sigma)
    else:
        return black_scholes_put(S, K, T, r, sigma)

In [6]:
S = 100.0 # spot
K = 100.0 # strike (at-the-money)
T = 1.0 # maturite 1 an
r = 0.05 # taux sans risque
sigma = 0.20 # volatilite brownienne
lam = 1.0 # en moyenne 1 saut par an

# Bscholes sans sauts
bs_call = black_scholes_call(S, K, T, r, sigma)
bs_put  = black_scholes_put(S, K, T, r, sigma)
print(f"\nBlack-Scholes de reference :")
print(f"  Call ATM : {bs_call:.4f}")
print(f"  Put  ATM : {bs_put:.4f}")


Black-Scholes de reference :
  Call ATM : 10.4506
  Put  ATM : 5.5735


# PARTIE 1 : Sauts discrets dans {a, b}

In [7]:
def prix_sauts_discrets(S, K, T, r, sigma, lam, a, b, p,
                        option_type='call', N=30):
    """
    Prix d'une option europeenne avecsauts dans {a, b}. Params :
    S     : prix initial du sous-jacent
    K     : strike
    T     : maturite
    r     : taux sans risque
    sigma : volatilite brownienne
    lam   : intensite du processus de Poisson
    a, b  : valeurs des sauts relatifs (U_j in {a, b})
    """
    # Esperance des sauts et intensites decomposees
    E_U1    = p * a + (1 - p) * b
    lam_a  = lam * p
    lam_b  = lam * (1 - p)
    # Facteur de correction du spot
    correction = exp(-lam * T * E_U1)
    prix = 0.0

    for k in range(N + 1):
        # Poids de Poisson(lambda_a * T) en k
        poids_k = exp(-lam_a * T) * (lam_a * T)**k / factorial(k)
        for m in range(N + 1):
            poids_m = exp(-lam_b * T) * (lam_b * T)**m / factorial(m)
            S_km = S * correction * (1 + a)**k * (1 + b)**m
            # Prix Black-Scholes avec ce spot modifie
            bs_km = black_scholes(S_km, K, T, r, sigma, option_type)
            prix += poids_k * poids_m * bs_km

    return prix

In [8]:
print("\nPARTIE 1 : Sauts discrets U_1 in {a, b}")
a = 0.10
b = -0.10
p = 0.5   # equiprobable

call_discret = prix_sauts_discrets(S, K, T, r, sigma, lam, a, b, p, option_type='call', N=50)
put_discret  = prix_sauts_discrets(S, K, T, r, sigma, lam, a, b, p, option_type='put', N=50)

print(f"\nParametres : a={a}, b={b}, p={p}, lambda={lam}")
print(f"  E[U1] = {p*a + (1-p)*b:.4f}")
print(f"  Call ATM : {call_discret:.4f}")
print(f"  Put  ATM : {put_discret:.4f}")

parite = S - K * exp(-r * T)
print(f"  Parite call-put theorique C-P = {parite:.4f}")
print(f"  Parite call-put obtenue   C-P = {call_discret - put_discret:.4f}")

# Effet de l'intensite
print("\nEffet de l'intensite lambda :")
print(f"  {'lambda':>8} | {'Call':>10} | {'Put':>10}")
print(f"  {'-'*8}-+-{'-'*10}-+-{'-'*10}")
for l in [0.0, 0.5, 1.0, 2.0, 5.0]:
    if l == 0.0:
        c = bs_call
        pu = bs_put
    else:
        c  = prix_sauts_discrets(S, K, T, r, sigma, l, a, b, p, option_type='call', N=25)
        pu = prix_sauts_discrets(S, K, T, r, sigma, l, a, b, p, option_type='put',  N=25)
    print(f"  {l:>8.1f} | {c:>10.4f} | {pu:>10.4f}")


PARTIE 1 : Sauts discrets U_1 in {a, b}

Parametres : a=0.1, b=-0.1, p=0.5, lambda=1.0
  E[U1] = 0.0000
  Call ATM : 11.3307
  Put  ATM : 6.4537
  Parite call-put theorique C-P = 4.8771
  Parite call-put obtenue   C-P = 4.8771

Effet de l'intensite lambda :
    lambda |       Call |        Put
  ---------+------------+-----------
       0.0 |    10.4506 |     5.5735
       0.5 |    10.9017 |     6.0246
       1.0 |    11.3307 |     6.4537
       2.0 |    12.1332 |     7.2561
       5.0 |    14.2181 |     9.3411


# PARTIE 2 : Sauts log-normaux

In [10]:
def prix_sauts_lognormaux(S, K, T, r, sigma, lam, m, sigma_J,
                           option_type='call', N=50):
    """
    Prix d'une option europeenne avec sauts log-normaux. Params :
    m : moyenne de g (log du saut + 1)
    sigma_J : ecart-type de g
    """
    # Esperance du saut (formule log-normale)
    mu_J = exp(m + 0.5 * sigma_J**2) - 1
    prix = 0.0

    for n in range(N + 1):
        poids_n = exp(-lam * T) * (lam * T)**n / factorial(n)
        # Volatilite augmentee des sauts
        sigma_n = sqrt(sigma**2 + n * sigma_J**2 / T)
        # Taux modifie
        r_n = r - lam * mu_J + n * m / T
        bs_n = black_scholes(S, K, T, r_n, sigma_n, option_type)
        prix += poids_n * bs_n

    return prix

In [13]:
print("PARTIE 2 : Sauts log-normaux U_1 = e^g - 1, g ~ N(m, sigma_J^2)")


m = -0.10 # sauts legerement baissiers en moyenne
sigma_J = 0.20 # volatilite des sauts

mu_J = exp(m + 0.5 * sigma_J**2) - 1
print(f"\nParametres : m={m}, sigma_J={sigma_J}, lambda={lam}")
print(f"  mu_J = E[U1] = {mu_J:.4f}")

call_logn = prix_sauts_lognormaux(S, K, T, r, sigma, lam, m, sigma_J, option_type='call', N=50)
put_logn  = prix_sauts_lognormaux(S, K, T, r, sigma, lam, m, sigma_J, option_type='put',  N=50)

print(f"  Call ATM : {call_logn:.4f}")
print(f"  Put  ATM : {put_logn:.4f}")

print(f"  Parite call-put theorique C-P = {parite:.4f}")
print(f"  Parite call-put obtenue   C-P = {call_logn - put_logn:.4f}")

# Effet de la volatilite des sauts
print("\nEffet volatilite des sauts:")
print(f"  {'sigma_J':>8} | {'Call':>10} | {'Put':>10}")
print(f"  {'-'*8}-+-{'-'*10}-+-{'-'*10}")
for sj in [0.0, 0.10, 0.20, 0.30, 0.50]:
    if sj == 0.0:
        c  = bs_call
        pu = bs_put
    else:
        c  = prix_sauts_lognormaux(S, K, T, r, sigma, lam, m, sj, option_type='call', N=50)
        pu = prix_sauts_lognormaux(S, K, T, r, sigma, lam, m, sj, option_type='put',  N=50)
    print(f"  {sj:>8.2f} | {c:>10.4f} | {pu:>10.4f}")

# Effet de la maturite
print("\nEffet de la maturite:")
print(f"  {'T':>8} | {'Call':>10} | {'Put':>10}")
print(f"  {'-'*8}-+-{'-'*10}-+-{'-'*10}")
for t in [0.25, 0.5, 1.0, 2.0]:
    c  = prix_sauts_lognormaux(S, K, t, r, sigma, lam, m, sigma_J, option_type='call', N=50)
    pu = prix_sauts_lognormaux(S, K, t, r, sigma, lam, m, sigma_J, option_type='put',  N=50)
    print(f"  {t:>8.2f} | {c:>10.4f} | {pu:>10.4f}")

PARTIE 2 : Sauts log-normaux U_1 = e^g - 1, g ~ N(m, sigma_J^2)

Parametres : m=-0.1, sigma_J=0.2, lambda=1.0
  mu_J = E[U1] = -0.0769
  Call ATM : 12.7917
  Put  ATM : 10.6439
  Parite call-put theorique C-P = 4.8771
  Parite call-put obtenue   C-P = 2.1479

Effet volatilite des sauts:
   sigma_J |       Call |        Put
  ---------+------------+-----------
      0.00 |    10.4506 |     5.5735
      0.10 |    11.4800 |     7.9965
      0.20 |    12.7917 |    10.6439
      0.30 |    14.1984 |    14.3642
      0.50 |    16.4039 |    24.7853

Effet de la maturite:
         T |       Call |        Put
  ---------+------------+-----------
      0.25 |     5.7681 |     5.2267
      0.50 |     8.6406 |     7.5609
      1.00 |    12.7917 |    10.6439
      2.00 |    18.7415 |    14.4919
